# SPEC 생성 평가 개정의 판단 재료

초안(`docs/drafts/spec-generation-eval.md`)의 남은 결정 사항 1(핵심 답변 포함도의 위치), 2(종합 지표 구성, 과잉 거부 허용폭 δ_r와 분모), 3(최소 개선 폭 δ)을 정할 재료다. 새 API 호출 없이 S24 반복 실행(판정용 앞 100질의, 생성 3회 × judge `mixed`)과 S25 베이스라인(330질의)으로 계산한다. 기준 설정 출력만 쓰고 후보 설정끼리 비교하지 않는다(초안 결정 사항 1의 제약). 결정은 적지 않는다.

In [1]:
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd

from rag.eval.bootstrap import paired_bootstrap
from rag.eval.metrics import load_gold
from rag.gen_eval.common import read_jsonl
from rag.index import load_cfg

In [2]:
ROOT = Path.cwd().parent
JUDGE = ROOT / "data/gen_eval/judge"
STEM = "hybrid-cond-filter-law-ms256-k20-ce-v2-dev-full"
SETTING = "deepseek_go_mixed-j_v8"
GEN_REPS = [0, 1, 2]
BASELINE = JUDGE / f"{STEM}-{SETTING}-n330.jsonl"
N_JUDGED = 330  # 판정용 질의 수(D-50)

In [3]:
def tag(n):
    return f"-r{n}" if n else ""


def per_query(rows):
    """질의 단위 값. 계산하지 않은 기준(전체 거부, 참조 사용 불가, 형식 오류)은 NaN."""
    out = {}
    for r in rows:
        c = (r.get("item_eval") or {}).get("counts") or {}
        items = (r.get("item_eval") or {}).get("items") or []
        answerable = sum(i["state"] in ("있음", "일부") for i in items)
        over = c.get("over_refusal", np.nan)
        out[r["qid"]] = {
            "충실성": (r.get("faithfulness") or {}).get("score"),
            "답변 관련성": (r.get("relevance") or {}).get("score"),
            "핵심 답변 포함도": (r.get("inclusion") or {}).get("score"),
            "과잉 거부/판정 대상 항목": over / c["judged"] if c.get("judged") else np.nan,
            "과잉 거부/답이 있는 항목": over / answerable if answerable else np.nan,
            "과잉 거부 있는 질의(답이 있는 질의 중)": float(over > 0) if answerable else np.nan,
            "답이 청크에 있음": r.get("answer_in_chunk"),
        }
    df = pd.DataFrame.from_dict(out, orient="index")
    df["평균(충실성·관련성)"] = df[["충실성", "답변 관련성"]].mean(axis=1, skipna=False)
    df["평균(세 기준)"] = df[["충실성", "답변 관련성", "핵심 답변 포함도"]].mean(axis=1, skipna=False)
    return df


def half_width(a, b, groups, boot):
    """두 실행의 질의 단위 값 차이의 95% 구간 반폭과 공통 질의 수(회의 단위 paired bootstrap)."""
    both = a.notna() & b.notna()
    _, lo, hi = paired_bootstrap((a - b)[both], groups[both[both].index], boot.n_resamples,
                                 boot.seed, boot.alpha)
    return (hi - lo) / 2, int(both.sum())

## 데이터 로딩

In [4]:
cfg = load_cfg(["index.scope=full", "+exp=exp017"])
boot = cfg.eval.bootstrap
groups = load_gold(cfg, "dev-full")["conference_number"]
reps = {g: per_query(read_jsonl(JUDGE / f"{STEM}{tag(g)}-{SETTING}.jsonl")) for g in GEN_REPS}
base = per_query(read_jsonl(BASELINE))
print({g: len(v) for g, v in reps.items()}, "베이스라인", len(base))
print("반복끼리 질의가 같은가", all(reps[g].index.equals(reps[0].index) for g in GEN_REPS))

{0: 100, 1: 100, 2: 100} 베이스라인 330
반복끼리 질의가 같은가 True


## 결정 사항 3: 최소 개선 폭의 바닥(같은 설정의 반복 차이)

생성과 judge를 모두 다시 돌린 같은 설정 두 실행의 평균 차이 95% 구간 반폭이다(D-50과 같은 방식). 같은 설정끼리도 이만큼은 우연히 벌어지므로, 최소 개선 폭이 이보다 작으면 같은 설정을 다시 돌리기만 해도 구간이 그 폭을 넘나든다. 330질의 값은 100질의 값을 √(공통 질의 수 비)로 줄인 어림이다.

In [10]:
METRICS = ["충실성", "답변 관련성", "핵심 답변 포함도", "평균(충실성·관련성)", "평균(세 기준)",
           "과잉 거부/판정 대상 항목", "과잉 거부/답이 있는 항목", "과잉 거부 있는 질의(답이 있는 질의 중)"]
rows = []
for m in METRICS:
    hws, ns = zip(*[half_width(reps[a][m].astype(float), reps[b][m].astype(float), groups, boot)
                    for a, b in combinations(GEN_REPS, 2)], strict=True)
    n100 = np.mean(ns)
    n330 = base[m].notna().sum()
    rows.append({"지표": m, "반복 평균값": np.nanmean([reps[g][m].astype(float).mean() for g in GEN_REPS]),
                 "베이스라인(330)": base[m].astype(float).mean(), "공통 질의(100 중)": n100,
                 "반폭 h(100)": np.mean(hws), "계산 질의(330 중)": n330,
                 "반폭 어림(330)": np.mean(hws) * np.sqrt(n100 / n330)})
noise = pd.DataFrame(rows).set_index("지표")
noise.round(3)

,반복 평균값,베이스라인(330),공통 질의(100 중),반폭 h(100),계산 질의(330 중),반폭 어림(330)
지표,,,,,,
충실성,0.775,0.803,92.000,0.057,314,0.031
답변 관련성,0.903,0.871,94.667,0.039,321,0.021
핵심 답변 포함도,0.587,0.512,74.000,0.042,258,0.023
평균(충실성·관련성),0.840,0.843,92.000,0.034,314,0.019
평균(세 기준),0.763,0.744,72.333,0.030,254,0.016
과잉 거부/판정 대상 항목,0.012,0.024,96.000,0.022,307,0.012
과잉 거부/답이 있는 항목,0.016,0.034,73.000,0.029,236,0.016
과잉 거부 있는 질의(답이 있는 질의 중),0.018,0.038,73.000,0.034,236,0.019


## 결정 사항 2: 과잉 거부율의 분모

같은 과잉 거부 항목을 세도 분모에 따라 값의 크기와 반복 사이 흔들림이 다르다. 판정 대상 항목은 모호·회피를 뺀 항목, 답이 있는 항목은 judge가 정보 '있음'·'일부'로 본 항목, 질의 단위는 답이 있는 항목이 하나라도 있는 질의 중 과잉 거부가 하나라도 있는 질의의 비율이다.

In [11]:
over = noise.loc[[m for m in METRICS if m.startswith("과잉")]]
over.assign(**{"반폭/평균": over["반폭 어림(330)"] / over["베이스라인(330)"]}).round(3)

,반복 평균값,베이스라인(330),공통 질의(100 중),반폭 h(100),계산 질의(330 중),반폭 어림(330),반폭/평균
지표,,,,,,,
과잉 거부/판정 대상 항목,0.012,0.024,96.0,0.022,307,0.012,0.510
과잉 거부/답이 있는 항목,0.016,0.034,73.0,0.029,236,0.016,0.479
과잉 거부 있는 질의(답이 있는 질의 중),0.018,0.038,73.0,0.034,236,0.019,0.495


## 결정 사항 1: 핵심 답변 포함도의 위치

포함도는 정답 문서가 근거에 있고 참조 답변(`answer.comment`)에서 핵심 진술을 뽑을 수 있을 때만 계산된다. 계산되는 비율, 다른 두 기준과의 관계, 답이 청크에 있을 때와 없을 때의 차이를 본다.

In [12]:
inc = base["핵심 답변 포함도"].astype(float)
print("포함도 계산 질의", int(inc.notna().sum()), "/", N_JUDGED)
print("세 기준을 모두 계산한 질의", int(base["평균(세 기준)"].notna().sum()), "/", N_JUDGED)
base[["충실성", "답변 관련성", "핵심 답변 포함도"]].astype(float).corr(method="spearman").round(3)

포함도 계산 질의 258 / 330
세 기준을 모두 계산한 질의 254 / 330


,충실성,답변 관련성,핵심 답변 포함도
충실성,1.000,-0.020,0.303
답변 관련성,-0.020,1.000,-0.053
핵심 답변 포함도,0.303,-0.053,1.000


In [13]:
# 답이 근거 청크에 있다고 judge가 본 질의(answer_in_chunk)와 아닌 질의의 포함도
split = base.assign(포함도=inc).groupby(base["답이 청크에 있음"].map({True: "청크에 답 있음", False: "청크에 답 없음"}))
split["포함도"].agg(["count", "mean", lambda s: (s < 0.5).sum()]).rename(columns={"<lambda_0>": "0.5 미만"}).round(3)

,count,mean,0.5 미만
답이 청크에 있음,,,
청크에 답 없음,68,0.066,67
청크에 답 있음,190,0.672,42


In [14]:
# 기준별 질의 단위 반복 SD(생성 3회 사이). 생성·judge 잡음 분해는 35_01에 있다
sd = np.nanmean(np.nanstd(np.stack([reps[g][["충실성", "답변 관련성", "핵심 답변 포함도"]]
                                    .astype(float).to_numpy() for g in GEN_REPS]), axis=0, ddof=1), axis=0)
pd.Series(sd, index=["충실성", "답변 관련성", "핵심 답변 포함도"], name="질의 단위 반복 SD 평균").round(3)

C:\workspace\meet-rag\.venv\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


충실성          0.101
답변 관련성       0.058
핵심 답변 포함도    0.058
Name: 질의 단위 반복 SD 평균, dtype: float64